In [1]:
!pip install protobuf

In [2]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoModel, AutoTokenizer
from torch.utils.data import DataLoader

from model import MultilabelModel
from dataset import AbstractsDataset
from trainer import Trainer

MODEL = 'microsoft/deberta-v3-base'
MAX_SEQ_LEN = 350
BATCH_SIZE = 56
N_CLASSES = 155
N_CLASSES_GROUPED = 8

# Initialize data, train test split

In [5]:
train = pd.read_csv('../data/train/train_data.csv')
test = pd.read_csv('../data/test/test_data.csv')

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

d:\Progs\anaconda3\Lib\site-packages\transformers\convert_slow_tokenizer.py:560: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [6]:
train_dataset = AbstractsDataset(train, tokenizer, MAX_SEQ_LEN)
test_dataset = AbstractsDataset(test, tokenizer, MAX_SEQ_LEN)

In [7]:
train_params = {
    'batch_size': BATCH_SIZE,
    'shuffle': True,
    'num_workers': 0
}
test_params = {
    'batch_size': BATCH_SIZE,
    'shuffle': False,
    'num_workers': 0
}

train_dataloader = DataLoader(train_dataset, **train_params)
test_dataloader = DataLoader(test_dataset, **test_params)

# Loading & fitting pretrained model

In [8]:
config = {
    "n_classes": N_CLASSES,
    "dropout_rate": 0.1
}

model = MultilabelModel(
    MODEL,
    config=config
)

In [9]:
trainer_config = {
    "lr": 1e-5,
    "n_epochs": 2,
    "batch_size": BATCH_SIZE,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42,
}

t = Trainer(trainer_config)
t.fit(model, train_dataloader, val_dataloader)

Epoch 1/5


  0%|          | 0/13451 [00:00<?, ?it/s]

KeyboardInterrupt: 